In [2]:
import rsa 

Salvando as chaves públicas e privadas

In [3]:
def private_key_to_file(private_key, filepath):   
    with open(filepath, 'wb+') as f:
        pk = rsa.PrivateKey.save_pkcs1(private_key, format='PEM')
        f.write(pk)

In [4]:
def public_key_to_file(public_key, filepath):   
    with open(filepath, 'wb+') as f:
        pk = rsa.PublicKey.save_pkcs1(public_key, format='PEM')
        f.write(pk)

Criando chaves de Bob

In [10]:
(bob_pub, bob_priv) = rsa.newkeys(512)

private_key_to_file(bob_priv, "bob_priv.pem")
public_key_to_file(bob_pub, "bob_pub.pem")

Criando chaves de Alice

In [11]:
(alice_pub, alice_priv) = rsa.newkeys(512)

private_key_to_file(alice_priv, "alice_priv.pem")
public_key_to_file(alice_pub, "alice_pub.pem")

Agora vamos simular a situação onde Bob deseja enviar uma carta (um arquivo texto) para Alice. Ele deseja criptografar este arquivo texto. 

Bob precisa carregar a chave pública de Alice

In [34]:
with open('alice_pub.pem', mode='rb') as publicfile:
    keydata = publicfile.read()
  
alice_pubkey = rsa.PublicKey.load_pkcs1(keydata)

RSA só pode criptografar mensagens menores que a chave. 

A maneira mais comum de usar RSA com arquivos maiores usa uma cifra de bloco como AES ou DES3 para criptografar o arquivo com uma chave aleatória e, em seguida, criptografar a chave aleatória com RSA. É necessário enviar o arquivo criptografado junto com a chave criptografada para o destinatário.


In [ ]:
!pip install pycrypto

In [31]:
# from Crypto.Cipher import AES
import rsa.randnum
from Crypto.Cipher import AES
from Crypto.Util import Counter

# gera chave AES
aes_key = rsa.randnum.read_random_bits(128)

mode = AES.MODE_CTR
encryptor = AES.new(aes_key, mode, counter=Counter.new(128))

In [ ]:
with open('lorem.txt', mode='rb') as filebob:
    message = filebob.read()

print(message)

ciphermessage = encryptor.encrypt(message)

print(ciphermessage)
with open("message.aes", 'wb+') as f:
    f.write(ciphermessage)

Agora Bob deverá criptografar a chave aes com a chave pública de Alice

In [36]:
encrypted_aes_key = rsa.encrypt(aes_key, alice_pubkey)

with open("encrypted_aes_key.pem", 'wb+') as f:
    f.write(encrypted_aes_key)

Bob envia a mensagem para Alice e a chave aes, ambos criptografados. 

In [ ]:
# Alice carrega o arquivo criptografado por Bob
with open('message.aes', mode='rb') as filebob:
    message = filebob.read()

print(message)

# Alice carrega a chave aes criptografada por Bob
with open('encrypted_aes_key.pem', mode='rb') as filebob:
    key = filebob.read()

# Alice carrega a sua chave privada
with open('alice_priv.pem', mode='rb') as privatefile:
    keydata = privatefile.read()

alice_privkey = rsa.PrivateKey.load_pkcs1(keydata)

# Alice usa a usa chave privada para descriptografar a chave aes
aeskey = rsa.decrypt(key, alice_privkey)

# Alice usa a chave aes para descriptografar o arquivo (mensagem de Bob)
decryptor = AES.new(aeskey, mode, counter=Counter.new(128))
m = decryptor.decrypt(message)
print(m)

Fonte: https://stuvel.eu/python-rsa-doc/usage.html